In [1]:

!pip install transformers torch accelerate -q

print("✅ Packages installed successfully!")

✅ Packages installed successfully!


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

# Check for GPU availability (Colab feature)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Running on: {str(device).upper()}")
print(f"🔋 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📦 GPU Model: {torch.cuda.get_device_name(0)}")

✅ Running on: CPU
🔋 GPU Available: False


In [4]:

MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"🔄 Loading model: {MODEL_NAME}")
print("⏳ This may take 2-5 minutes on first run (downloading ~1.5GB)...")

# Load tokenizer (converts text ↔ tokens)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load pre-trained causal language model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Move model to GPU if available for faster inference
model.to(device)

print("✅ Model loaded successfully!")
print(f"📊 Model parameters: ~762 million")

🔄 Loading model: microsoft/DialoGPT-medium
⏳ This may take 2-5 minutes on first run (downloading ~1.5GB)...


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded successfully!
📊 Model parameters: ~762 million


In [5]:

class DialoGPTChatbot:
    """
    Interactive chatbot using Hugging Face DialoGPT model.
    Features: context tracking, configurable generation, graceful exit.
    """

    def __init__(self, model, tokenizer, max_history=5):
        """
        Initialize chatbot with model, tokenizer, and history limit.

        Args:
            model: Pre-trained DialoGPT model
            tokenizer: Corresponding Hugging Face tokenizer
            max_history: Number of turns to keep in context window
        """
        self.model = model
        self.tokenizer = tokenizer
        self.max_history = max_history
        self.chat_history_ids = None

    def generate_response(self, user_input, max_length=1000, temperature=0.7, top_k=50):
        """
        Generate AI response using transformer model with controlled sampling.

        Args:
            user_input: User's message (string)
            max_length: Max tokens for response (prevents runaway generation)
            temperature: Creativity control (0.0=deterministic, 1.0=creative)
            top_k: Sample from top-k probable tokens (reduces nonsense)

        Returns:
            str: Generated bot response
        """
        # Encode input with EOS token (required by DialoGPT architecture) [[3]]
        input_ids = self.tokenizer.encode(
            user_input + self.tokenizer.eos_token,
            return_tensors='pt'
        ).to(device)  # Move to GPU/CPU

        # Concatenate with conversation history for context-aware responses
        if self.chat_history_ids is not None:
            bot_input_ids = torch.cat([self.chat_history_ids, input_ids], dim=-1)
        else:
            bot_input_ids = input_ids

        # Generate response using optimized parameters [[1]][[3]]
        self.chat_history_ids = self.model.generate(
            bot_input_ids,
            max_length=max_length,
            do_sample=True,           # Enable sampling for natural dialogue
            temperature=temperature,  # Balance creativity vs coherence
            top_k=top_k,             # Focus on high-probability tokens
            pad_token_id=self.tokenizer.eos_token_id,
            num_return_sequences=1,
            no_repeat_ngram_size=3,  # Reduce repetitive phrases
            early_stopping=True
        )

        # Decode ONLY the new tokens (bot's response), skipping history
        response = self.tokenizer.decode(
            self.chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        return response.strip()

    def reset(self):
        """Clear conversation history to start fresh"""
        self.chat_history_ids = None
        print("🔄 Conversation history cleared.")

In [6]:

def start_chatbot():
    """
    Run interactive chat session with user input handling and exit conditions.
    """
    # Initialize chatbot instance
    chatbot = DialoGPTChatbot(model, tokenizer)

    # Welcome banner
    print("\n" + "╔" + "═"*58 + "╗")
    print("║" + "🤖 AI Assistant - Powered by DialoGPT".center(58) + "║")
    print("╠" + "═"*58 + "╣")
    print("║ 💬 Type your message and press Enter".ljust(58) + "║")
    print("║ 🚪 Type 'exit' or 'quit' to end conversation".ljust(58) + "║")
    print("║ 🔄 Type 'reset' to clear history".ljust(58) + "║")
    print("╚" + "═"*58 + "╝\n")

    # Main conversation loop
    while True:
        try:
            # Get user input
            user_input = input(">> You: ").strip()

            # Exit conditions (case-insensitive)
            if user_input.lower() in ['exit', 'quit', 'bye', 'goodbye', 'see you']:
                print("\n🤖 Chatbot: Thank you for chatting! Have a wonderful day! 👋✨\n")
                break

            # Reset command
            if user_input.lower() == 'reset':
                chatbot.reset()
                print("🤖 Chatbot: History cleared! How can I help you now?\n")
                continue

            # Skip empty inputs
            if not user_input:
                print("🤖 Chatbot: Please type a message.\n")
                continue

            # Generate and display response
            print("🤖 Chatbot: ", end="", flush=True)
            response = chatbot.generate_response(user_input)
            print(f"{response}\n")

        except KeyboardInterrupt:
            print("\n\n🤖 Chatbot: Conversation interrupted. Goodbye! 👋\n")
            break
        except Exception as e:
            print(f"\n⚠️ Error: {str(e)}")
            print("🤖 Chatbot: Let's try that again.\n")
            continue

In [10]:


if __name__ == "__main__":
    start_chatbot()


╔══════════════════════════════════════════════════════════╗
║           🤖 AI Assistant - Powered by DialoGPT           ║
╠══════════════════════════════════════════════════════════╣
║ 💬 Type your message and press Enter                     ║
║ 🚪 Type 'exit' or 'quit' to end conversation             ║
║ 🔄 Type 'reset' to clear history                         ║
╚══════════════════════════════════════════════════════════╝

>> You: what is your name?
🤖 Chatbot: It's in the title

>> You: list some exercises that can done in home itself
🤖 Chatbot: I think this is something you can do in a lab

>> You: what?
🤖 Chatbot: You can make apps that run in the browser, but it is still a lab.

>> You: quit

🤖 Chatbot: Thank you for chatting! Have a wonderful day! 👋✨

